<div dir="rtl" align="right">

# رسمُ محاولاتِ EEG الخامِ من MOABB

**مجموعةُ البياناتِ**: BNCI2014-001 (تَخيّلٌ حركيٌّ)
**المُشاركُ**: 1
**النموذجُ**: MotorImagery (n_classes=2)
**القنواتُ**: 22 EEG
**معدّلُ أخذِ العيناتِ**: 250 Hz

---

## نظرةٌ عامّةٌ

هذا الدفترُ يحمّلُ حقبَ التَخيّلِ الحركيِّ من BNCI2014-001 ويُصوّرُ محاولاتِ EEG الخامَ لِتَخيّلِ اليدِ اليسرى واليمنى، مُظهراً جميعَ القنواتِ الـ 22 بِإزاحاتٍ عموديةٍ لِأولِ 5 ثوانٍ.

## ماذا يَفعلُ هذا الدفترُ

- يحمّلُ BNCI2014-001 لِلمُشاركِ 1 عبرَ MOABB
- يَستخرجُ الحقبَ بِنموذجِ MotorImagery
- يَختارُ محاولةً واحدةً لِليدِ اليسرى وأخرى لِليمنى
- يَرسمُ المحاولتينِ كَآثارٍ مُتعددةِ القنواتِ بِإزاحاتٍ

## المُخرجاتُ المُتوقّعةُ

- لوحتانِ: تَخيّلُ اليدِ اليسرى واليمنى
- 22 أثراً لِلقنواتِ مُرتّبةٌ عمودياً بِإزاحاتٍ
- 5 ثوانٍ من البياناتِ لِكلِّ محاولةٍ
- أنماطٌ مميّزةٌ فوقَ قنواتِ القشرةِ الحركيةِ (C3, C4, Cz)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| dataset | BNCI2014_001 | مجموعةُ بياناتِ تَخيّلٍ حركيٍّ من MOABB |
| subjects | [1] | المُشاركُ 1 فقط |
| n_classes | 2 | فئاتُ نموذجِ MotorImagery |
| plot_duration | 5 s | ثوانٍ من البياناتِ لِلعرضِ |
| offset | auto | التباعدُ العموديُّ بينَ القنواتِ |


</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

MOABB تُنزّلُ البياناتِ تلقائياً عندَ أولِ استخدامٍ (~44 ميجابايت لِلمُشاركِ 1). التشغيلاتُ اللاحقةُ تَستخدمُ البياناتِ المُخزّنةَ.


</div>


In [ ]:
from moabb.datasets import BNCI2014_001
ds = BNCI2014_001()
sessions = ds.get_data(subjects=[1])
subject_key = list(sessions.keys())[0]
session_dict = sessions[subject_key]
n_sessions = len(session_dict)
n_runs = len(next(iter(session_dict.values())))
first_run = next(iter(next(iter(session_dict.values())).values()))
n_channels_raw = len(first_run.ch_names)
sfreq = first_run.info['sfreq']
print(f'Subject 1: {n_sessions} sessions, {n_runs} runs/session')
print(f'Raw channels: {n_channels_raw}, Sampling rate: {sfreq} Hz')
print(f'Channel names: {first_run.ch_names}')



In [ ]:
from moabb.paradigms import MotorImagery
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=ds, subjects=[1])
print(f'X shape: {X.shape}  (n_trials, n_channels, n_samples)')
print(f'Labels shape: {labels.shape}')
print(f'Meta shape: {meta.shape}')



<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

نَطبعُ أشكالَ الحقبِ والوسومَ المُتاحةَ.

</div>


In [ ]:
import numpy as np
unique_labels, counts = np.unique(labels, return_counts=True)
print(f'Epoch channels: {X.shape[1]}')
print(f'Epoch samples: {X.shape[2]}')
print(f'Epoch duration: {X.shape[2] / sfreq:.2f} s')
print(f'Unique labels: {list(unique_labels)}')
print(f'Trials per class: {dict(zip(unique_labels, counts))}')
print(f'Total trials: {X.shape[0]}')



<div dir="rtl" align="right">

## 4. تطبيقُ التحليلِ

نَختارُ محاولةً واحدةً لِليدِ اليسرى وأخرى لِليمنى ونُجهّزُ محورَ الزمنِ لِلرسمِ.


</div>


In [ ]:
import numpy as np
sfreq = 250
n_samples = X.shape[2]
plot_samples = min(int(5 * sfreq), n_samples)
time = np.arange(plot_samples) / sfreq
left_idx = np.where(labels == 'left_hand')[0][0]
right_idx = np.where(labels == 'right_hand')[0][0]
left_trial = X[left_idx, :, :plot_samples]
right_trial = X[right_idx, :, :plot_samples]
print(f'Left hand trial index: {left_idx}')
print(f'Right hand trial index: {right_idx}')
print(f'Plotting {plot_samples} samples ({plot_samples/sfreq:.1f} s)')



<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- لوحتانِ تُظهرانِ تَخيّلَ اليدِ اليسرى واليمنى
- 22 قناةً مُرتّبةٌ بِإزاحاتٍ عموديةٍ
- اختلافاتٌ في أنماطِ السعةِ بينَ الحالتينِ
- قنواتُ القشرةِ الحركيةِ (C3, C4) قد تُظهرُ نشاطاً مُختلفاً بِحسبِ الحالةِ



</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
n_channels = X.shape[1]
offset_step = 1.2 * np.max(np.abs(X))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Left hand motor imagery', 'Right hand motor imagery'))
for ch in range(n_channels):
    fig.add_trace(go.Scatter(x=time, y=left_trial[ch] + ch * offset_step,
                             mode='lines', line=dict(width=0.6),
                             showlegend=False), row=1, col=1)
for ch in range(n_channels):
    fig.add_trace(go.Scatter(x=time, y=right_trial[ch] + ch * offset_step,
                             mode='lines', line=dict(width=0.6),
                             showlegend=False), row=2, col=1)
fig.update_xaxes(title_text='Time (s)', row=2, col=1)
fig.update_layout(height=800, title_text='MOABB BNCI2014-001 - Motor Imagery Trials',
                  xaxis_range=[0, plot_samples/sfreq])
fig.show()



<div dir="rtl" align="right">

## خلاصةٌ

- حقبُ MOABB تَحتوي على محاولاتِ EEG مُتعددةِ القنواتِ جاهزةٍ لِلتصويرِ
- تَخيّلُ اليدِ اليسرى واليمنى يُظهرانِ أنماطَ قنواتٍ مميّزةً
- الإزاحاتُ العموديةُ تُسهّلُ فحصَ جميعِ القنواتِ الـ 22 في آنٍ واحدٍ
- قنواتُ القشرةِ الحركيةِ (C3, C4) أساسيةٌ لِتصنيفِ التَخيّلِ الحركيِّ



</div>
